In [ ]:
# 0) Import ndp endpoint library
from ndp_ep import APIClient, remote_func

In [ ]:
# 1) Instantiate ndp ep client with API base URL and user access token.
client = APIClient(
    # base_url="http://ndp-test-211.chpc.utah.edu/api/test",
    base_url="http://localhost:8002",
    token="eyJhbGciOiJSUzI1NiIsInR5cCIgOiAiSldUIiwia2lkIiA6ICJlMXVpdExqOFNzbHdkWTFwYmZIRThkVFFPa0VpaGpmdTlRV3psSENtNGI4In0.eyJleHAiOjE3NjczOTU2NTYsImlhdCI6MTc2NzM3NzY1NiwianRpIjoiZTRlYzYyMmYtOTU1Yy00ODRhLTlkYzEtOTJjYWQ5OGQ1ODIzIiwiaXNzIjoiaHR0cHM6Ly9pZHAtdGVzdC5uYXRpb25hbGRhdGFwbGF0Zm9ybS5vcmcvcmVhbG1zL05EUCIsImF1ZCI6ImFjY291bnQiLCJzdWIiOiI0NjVkYTJjNC1mNTE0LTRhZWItOWU5Zi1hMzQwNDQ3MDk2ZWMiLCJ0eXAiOiJCZWFyZXIiLCJhenAiOiJORFBfQ0xJRU5UIiwic2Vzc2lvbl9zdGF0ZSI6Ijg2ZDMwN2YzLWJjNmQtNDNiNy1hZjgxLTdkZDU2MzkxOGQ2ZCIsImFjciI6IjEiLCJhbGxvd2VkLW9yaWdpbnMiOlsiKiJdLCJyZWFsbV9hY2Nlc3MiOnsicm9sZXMiOlsiZGVmYXVsdC1yb2xlcy1uZHAiLCJvZmZsaW5lX2FjY2VzcyIsInVtYV9hdXRob3JpemF0aW9uIl19LCJyZXNvdXJjZV9hY2Nlc3MiOnsiYWNjb3VudCI6eyJyb2xlcyI6WyJtYW5hZ2UtYWNjb3VudCIsIm1hbmFnZS1hY2NvdW50LWxpbmtzIiwidmlldy1wcm9maWxlIl19fSwic2NvcGUiOiJwcm9maWxlIGVtYWlsIiwic2lkIjoiODZkMzA3ZjMtYmM2ZC00M2I3LWFmODEtN2RkNTYzOTE4ZDZkIiwiZW1haWxfdmVyaWZpZWQiOnRydWUsIm5hbWUiOiJ5dXRpYW4gcWluIiwicHJlZmVycmVkX3VzZXJuYW1lIjoieXV0aWFuQGVtYWlsLmNvbSIsImdpdmVuX25hbWUiOiJ5dXRpYW4iLCJmYW1pbHlfbmFtZSI6InFpbiIsImVtYWlsIjoieXV0aWFuQGVtYWlsLmNvbSJ9.HcgT1XlaHONsw9eoKB4o64OYsq5RMsLfpkTgTiluKEVRJM5N9MRIUlDZqiFNd1n7vAObpo5eGM1bhS7UF7FciVVA-shXCQAcYB-kydd-4-yelO2zilyJ2O7wIEX1Vsrg78ASyoYRm-ND2A2mLb543kgQPM4qgNHZNjmcqT0-RRSdL3j5Ig08h7JD3cVBMMAxPhxWD_xV-vB_viXPlCVMVZwh4B378Fk3OgSTPVg1syOdtfWCFN9x38KyzbhhO56YcHA8-P87p4h6Nf9w7KGBjuI6XmRvMFRwH_vIFqSCGnfhzZr4rnwjn9ncvqZQvXVZZHBxBiy8RG_50A_FQrxIqA"
)
print("client initialized")

In [ ]:
# 2) Provision the remote environment with input_requirements.txt
client.setup_rexec_environment(
    requirements="input_requirements.txt"
)

In [ ]:
# Search dataset through ep client
import json
search_terms = ["earthscope_stations"]
results = client.search_datasets(terms=search_terms, server="global")

if results: print(json.dumps(results, indent=2, ensure_ascii=False))

In [ ]:
# Visualize the EarthScope station coverage
# import csv
# import io
# import urllib.request
from IPython.display import Image

if not results:
    raise ValueError("No search results found")
dataset_url = results[0]["resources"][0]["url"]
print(f"Using dataset: {dataset_url}")

@remote_func
def plot_earthscope_stations(csv_url: str, sample_size: int = 800):
    """Download the station CSV, plot lat/lon, and return PNG bytes plus counts."""
    import csv
    import io
    import urllib.request
    from collections import Counter
    import matplotlib.pyplot as plt

    with urllib.request.urlopen(csv_url) as resp:
        text = resp.read().decode("utf-8").splitlines()

    reader = csv.reader(text)
    next(reader, None)  # skip header

    lats, lons, nets, statuses = [], [], [], []
    for row in reader:
        if len(row) < 10:
            continue
        try:
            lat = float(row[1])
            lon = float(row[2])
        except ValueError:
            continue

        lats.append(lat)
        lons.append(lon)
        nets.append(row[8] if len(row) > 8 else "unknown")
        statuses.append(row[9] if len(row) > 9 else "unknown")

        if sample_size and len(lats) >= sample_size:
            break

    net_counts = Counter(nets)
    status_counts = Counter(statuses)
    unique_nets = sorted(net_counts)
    colors = plt.cm.tab20(range(len(unique_nets)))

    fig, ax = plt.subplots(figsize=(10, 6))
    for color, net in zip(colors, unique_nets):
        xs = [lo for lo, n in zip(lons, nets) if n == net]
        ys = [la for la, n in zip(lats, nets) if n == net]
        ax.scatter(xs, ys, s=12, color=color, label=net, alpha=0.75)

    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_title("EarthScope GNSS stations by network")
    ax.grid(True, linestyle="--", alpha=0.3)
    if len(unique_nets) <= 12:
        ax.legend(loc="lower left", bbox_to_anchor=(1.02, 0), frameon=False)

    fig.tight_layout()
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=150)
    plt.close(fig)
    buf.seek(0)
    return {"image": buf.getvalue(), "net_counts": dict(net_counts), "status_counts": dict(status_counts)}


viz = plot_earthscope_stations(dataset_url, 800)
print("Stations by network:", viz["net_counts"])
print("Stations by status:", viz["status_counts"])
Image(viz["image"])


In [ ]:
@remote_func
def hello_world():
    return "hello world!"

In [ ]:
ret = hello_world()
print(f"hello_world(): {ret}")

In [ ]:
@remote_func
def add(a, b):
    return a + b

In [ ]:
ret = add(5, 2)
print(f"add(5, 2): {ret}")